# PCA on median flux profiles

Run from repo root after CHRR sampling. Excludes unsupported reactions.

In [ ]:
import cobra
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

MODEL_DIR = "model/contextualized_models"
STAGES = ["egg_c", "L1_c", "L2_c", "L3_c", "L4f_c", "L4m_c", "Am_c", "Af_c", "L4mf_union_c"]

print(f"Models: {len(STAGES)}")
print(f"Stages: {', '.join(STAGES)}")

In [ ]:
# Load median flux profiles
medians = {}
for stage in STAGES:
    df = pd.read_csv(f"{stage}_median.csv").set_index("rxnID")["medianFlux"]
    medians[stage] = df
    print(f"{stage}: {len(df)} reactions")

# Find shared reactions across all models
shared = set.intersection(*(set(m.index) for m in medians.values()))
print(f"\nShared reactions: {len(shared)}")

In [ ]:
# Load subsystem info from models and filter unsupported
subsystem = {}
for stage in STAGES:
    model = cobra.io.read_sbml_model(f"{MODEL_DIR}/{stage}.xml")
    for rxn in model.reactions:
        subsystem.setdefault(rxn.id, rxn.subsystem)

# Exclude unsupported reactions
shared = sorted([r for r in shared if subsystem.get(r, "") != "Unsupported"])
print(f"Shared reactions (after unsupported exclusion): {len(shared)}")

In [ ]:
# Build data matrix and standardize
data = pd.DataFrame({s: medians[s].loc[shared] for s in STAGES}).T
print(f"Data shape: {data.shape} (models x reactions)")

scaled = StandardScaler().fit_transform(data)
print(f"Scaled shape: {scaled.shape}")

In [ ]:
# Run PCA
pca = PCA(svd_solver="full")
scores = pca.fit_transform(scaled)

print(f"PC1: {pca.explained_variance_ratio_[0]:.3f}")
print(f"PC2: {pca.explained_variance_ratio_[1]:.3f}")
print(f"Sum: {sum(pca.explained_variance_ratio_[:2]):.3f}")

In [ ]:
# Save outputs
pd.DataFrame(scores, index=STAGES, columns=[f"PC{i+1}" for i in range(scores.shape[1])]).to_csv("pca_scores.csv")
pd.Series(pca.explained_variance_ratio_, index=[f"PC{i+1}" for i in range(scores.shape[1])]).to_csv("pca_explained_variance.csv", header=["explained_variance_ratio"])

loadings = pd.DataFrame(pca.components_.T, index=shared, columns=[f"PC{i+1}" for i in range(scores.shape[1])])
loadings.insert(0, "subsystem", [subsystem.get(r, "") for r in shared])
loadings.to_csv("pca_loadings.csv", index_label="rxnID")

print("✓ Saved: pca_scores.csv, pca_explained_variance.csv, pca_loadings.csv")